# 변경사항 감지

새로 수집한 데이터와 이전 데이터를 비교하여 새로 추가된 필드만 감지합니다.

## Parameters

In [ ]:
# Papermill parameters (DAG에서 전달받음)
env = "stg"
dt = "2024-08-07"
version_date = "20240807"
gcs_bucket_name = "air-airflow-stg"

## 1. 라이브러리 및 설정 로드

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional, Set, Tuple
from google.cloud import storage

print(f"환경: {env}")
print(f"처리 날짜: {dt}")
print(f"버전 날짜: {version_date}")

## 2. 파일 탐지 함수 정의

In [ ]:
# GCS에서 최근 2개 파일 찾기
storage_client = storage.Client()
bucket = storage_client.bucket(gcs_bucket_name)

# product_meta_raw 폴더에서 mobile_plan_info 파일들 찾기
prefix = "product_meta_raw/"
blobs = list(bucket.list_blobs(prefix=prefix))

# mobile_plan_info 파일들만 필터링하고 날짜순 정렬
mobile_plan_files = []
pattern = re.compile(r'mobile_plan_info_(\d{8})\.json$')

for blob in blobs:
    match = pattern.search(blob.name)
    if match:
        date_str = match.group(1)
        mobile_plan_files.append((date_str, blob.name))

# 날짜순 정렬 (최신순)
mobile_plan_files.sort(key=lambda x: x[0], reverse=True)

print(f"🔍 GCS에서 탐지된 mobile_plan_info 파일들:")
for date_str, blob_name in mobile_plan_files[:5]:  # 최근 5개만 표시
    print(f"  - {blob_name} (날짜: {date_str})")

# 비교할 파일 결정 (최신 2개)
if len(mobile_plan_files) < 2:
    print("⚠️ 비교할 파일이 부족합니다. 최소 2개의 파일이 필요합니다.")
    if len(mobile_plan_files) == 1:
        print(f"   현재 파일만 있음: {mobile_plan_files[0][1]}")
        print("   → 첫 번째 실행이므로 모든 데이터를 새로운 것으로 간주합니다.")
        current_blob_name = mobile_plan_files[0][1]
        previous_blob_name = None
    else:
        raise ValueError("비교할 데이터 파일이 없습니다.")
else:
    current_blob_name = mobile_plan_files[0][1]  # 가장 최신
    previous_blob_name = mobile_plan_files[1][1]  # 두 번째 최신

print(f"\n📂 비교 대상:")
print(f"  - 이전 파일: {previous_blob_name or '없음'}")
print(f"  - 현재 파일: {current_blob_name}")

## 3. 데이터 로드 및 구조 분석

In [ ]:
def load_json_from_gcs(blob_name: str) -> Optional[Dict[str, Any]]:
    """GCS에서 JSON 파일 로드"""
    try:
        blob = bucket.blob(blob_name)
        if not blob.exists():
            print(f"⚠️ GCS 파일이 존재하지 않음: {blob_name}")
            return None
        
        json_data = blob.download_as_text()
        return json.loads(json_data)
    except Exception as e:
        print(f"❌ GCS 파일 로드 실패 {blob_name}: {str(e)}")
        return None

def extract_product_list(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """JSON 데이터에서 상품 리스트 추출"""
    # 새로운 형식 (metadata + result_list)
    if 'result_list' in data:
        return data['result_list']
    # 기존 형식 (직접 배열)
    elif isinstance(data, list):
        return data
    # 기타 형식
    else:
        print(f"⚠️ 알 수 없는 데이터 형식: {list(data.keys()) if isinstance(data, dict) else type(data)}")
        return []

# 파일 로드
previous_data = None
current_data = None

# 이전 데이터 로드
if previous_blob_name:
    previous_data = load_json_from_gcs(previous_blob_name)
    if previous_data:
        previous_products = extract_product_list(previous_data)
        print(f"\n✅ 이전 데이터 로드 완료: {len(previous_products)}개 상품")
    else:
        print(f"\n❌ 이전 데이터 로드 실패: {previous_blob_name}")
        previous_products = []
else:
    print(f"\n📝 이전 데이터 없음 - 첫 번째 실행")
    previous_products = []

# 현재 데이터 로드
current_data = load_json_from_gcs(current_blob_name)
if current_data:
    current_products = extract_product_list(current_data)
    print(f"✅ 현재 데이터 로드 완료: {len(current_products)}개 상품")
    
    # 메타데이터 확인
    if 'metadata' in current_data:
        print(f"📋 현재 데이터 메타데이터: {current_data['metadata']}")
else:
    print(f"❌ 현재 데이터 로드 실패: {current_blob_name}")
    current_products = []

if not current_products:
    raise ValueError("현재 데이터가 없어서 비교할 수 없습니다.")

## 4. 상품 딕셔너리 변환 및 ID 매핑

In [ ]:
def get_product_dict(data: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """상품 리스트를 상품 ID 기준 딕셔너리로 변환"""
    product_dict = {}
    extraction_stats = {"success": 0, "fallback": 0, "failed": 0}
    
    for i, product in enumerate(data):
        product_id = None
        
        # MAP 상품 상세 API의 JSON 구조가 일관성이 부족함으로 인해 각 JSON 객체에서 고유한 상품 ID를 추출하는 로직을 3단계 시도로 구성 
        # 1차: mappedProductCode에서 상품 코드 추출
        try:
            mapped_code = product.get("managementInfo", {}).get("mappedProductCode", {})
            product_codes = mapped_code.get("productCode", {}).get("valueList", [])
            
            if product_codes and len(product_codes) > 0:
                product_id = str(product_codes[0])  # 첫 번째 코드 사용
                extraction_stats["success"] += 1
                
        except (KeyError, TypeError, AttributeError):
            pass
        
        # 2차: pmProductID 사용 (fallback)
        if not product_id:
            pm_id = product.get("pmProductID")
            if pm_id:
                product_id = str(pm_id)
                extraction_stats["fallback"] += 1
        
        # 3차: 기타 ID 필드들 시도
        if not product_id:
            for id_field in ["productId", "id", "productCode"]:
                if id_field in product and product[id_field]:
                    product_id = str(product[id_field])
                    extraction_stats["fallback"] += 1
                    break
        
        # 엄격한 검증: Product ID를 찾지 못하면 에러 발생
        if not product_id:
            extraction_stats["failed"] += 1
            available_keys = list(product.keys())[:10]  # 최대 10개 키만 표시
            raise ValueError(
                f"❌ Product ID 추출 실패 (상품 인덱스: {i})\n"
                f"   사용 가능한 키들: {available_keys}\n"
                f"   상품 데이터 일부: {str(product)[:200]}...\n"
                f"   모든 상품에 고유 ID가 있어야 합니다. 데이터를 확인해주세요."
            )
        
        # 중복 ID 검사
        if product_id in product_dict:
            raise ValueError(
                f"❌ 중복된 Product ID 발견: {product_id}\n"
                f"   이전 상품과 현재 상품(인덱스: {i})이 동일한 ID를 가지고 있습니다.\n"
                f"   데이터 무결성을 확인해주세요."
            )
        
        product_dict[product_id] = product
    
    return product_dict, extraction_stats

# 상품 딕셔너리 변환
try:
    previous_products_dict, prev_stats = get_product_dict(previous_products)
    current_products_dict, curr_stats = get_product_dict(current_products)
    
    print(f"\n📊 상품 ID 추출 결과:")
    print(f"  이전 데이터: 성공 {prev_stats['success']}, 대체 {prev_stats['fallback']}, 실패 {prev_stats['failed']}")
    print(f"  현재 데이터: 성공 {curr_stats['success']}, 대체 {curr_stats['fallback']}, 실패 {curr_stats['failed']}")

    print(f"\n🔄 상품 ID 기준 딕셔너리 변환 완료:")
    print(f"  - 이전: {len(previous_products_dict)}개 상품")
    print(f"  - 현재: {len(current_products_dict)}개 상품")

    if current_products_dict:
        sample_id = list(current_products_dict.keys())[0]
        print(f"\n📝 현재 데이터 샘플 ID: {sample_id}")
        sample_product = current_products_dict[sample_id]
        print(f"   샘플 상품 최상위 키: {list(sample_product.keys())[:10]}")

except ValueError as e:
    print(f"\n{str(e)}")
    raise  # 에러를 다시 발생시켜서 노트북 실행을 중단

## 5. 새 스키마 감지 및 결과 정리

In [ ]:
# 새로운 스키마/필드 감지 - 올바른 방식
print(f"\n=== 새 필드/스키마 변경사항 감지 시작 ===")

# ID 세트 분석
previous_ids = set(previous_products_dict.keys())
current_ids = set(current_products_dict.keys())

added_ids = current_ids - previous_ids
common_ids = previous_ids & current_ids
removed_ids = previous_ids - current_ids

print(f"상품 ID 변화:")
print(f"- 새로 추가된 상품: {len(added_ids)}개")
print(f"- 공통 상품: {len(common_ids)}개")
print(f"- 제거된 상품: {len(removed_ids)}개")

# 이전 데이터의 전체 스키마 추출 (모든 기존 상품의 필드들)
def extract_all_fields(products_dict: Dict[str, Dict[str, Any]], prefix: str = "") -> Set[str]:
    """상품 딕셔너리에서 모든 필드 경로를 추출"""
    all_fields = set()
    
    def extract_fields_recursive(obj: Any, path: str = ""):
        if isinstance(obj, dict):
            for key, value in obj.items():
                current_path = f"{path}.{key}" if path else key
                all_fields.add(current_path)
                extract_fields_recursive(value, current_path)
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                current_path = f"{path}[{i}]"
                all_fields.add(current_path)
                extract_fields_recursive(item, current_path)
    
    for product_id, product_data in products_dict.items():
        extract_fields_recursive(product_data, f"{prefix}{product_id}")
    
    return all_fields

# 이전 데이터의 전체 스키마 (기준점)
if previous_products_dict:
    previous_schema_fields = extract_all_fields(previous_products_dict, "product_")
    print(f"\n이전 데이터 전체 스키마: {len(previous_schema_fields)}개 필드 패턴")
else:
    previous_schema_fields = set()
    print(f"\n이전 데이터 없음 - 첫 번째 실행")

# 현재 데이터의 전체 스키마
current_schema_fields = extract_all_fields(current_products_dict, "product_")
print(f"현재 데이터 전체 스키마: {len(current_schema_fields)}개 필드 패턴")

# 진짜 새로운 스키마 패턴 찾기 (상품 ID 제외하고 패턴만 비교)
def normalize_field_path(field_path: str) -> str:
    """필드 경로에서 상품 ID 부분을 제거하여 스키마 패턴만 추출"""
    # product_XXXXX.field.name -> product_*.field.name
    import re
    return re.sub(r'product_[^.]+', 'product_*', field_path)

previous_schema_patterns = {normalize_field_path(f) for f in previous_schema_fields}
current_schema_patterns = {normalize_field_path(f) for f in current_schema_fields}

# 진짜 새로운 스키마 패턴들
new_schema_patterns = current_schema_patterns - previous_schema_patterns

print(f"\n스키마 패턴 분석:")
print(f"- 이전 스키마 패턴: {len(previous_schema_patterns)}개")
print(f"- 현재 스키마 패턴: {len(current_schema_patterns)}개")
print(f"- 진짜 새로운 스키마 패턴: {len(new_schema_patterns)}개")

# 새로운 스키마 패턴 샘플 출력
if new_schema_patterns:
    print(f"\n새로운 스키마 패턴 예시:")
    for pattern in list(new_schema_patterns)[:5]:
        print(f"  - {pattern}")

# 실제 새 필드 변경사항 수집 (새로운 스키마 패턴에 해당하는 것만)
all_new_field_changes = []

print(f"\n전체 {len(current_ids)}개 상품에서 새 스키마 패턴 기반으로 새 필드 검사 중...")

processed = 0
for product_id in current_ids:
    processed += 1
    
    if processed % 1000 == 0 or processed == len(current_ids):
        print(f"진행: {processed}/{len(current_ids)} ({processed/len(current_ids)*100:.1f}%)")
    
    current_product = current_products_dict[product_id]
    
    # 현재 상품의 모든 필드 추출
    current_product_fields = set()
    def extract_product_fields(obj: Any, path: str = ""):
        if isinstance(obj, dict):
            for key, value in obj.items():
                current_path = f"{path}.{key}" if path else key
                current_product_fields.add(current_path)
                extract_product_fields(value, current_path)
        elif isinstance(obj, list):
            for i, item in enumerate(obj):
                current_path = f"{path}[{i}]"
                current_product_fields.add(current_path)
                extract_product_fields(item, current_path)
    
    extract_product_fields(current_product, f"product_{product_id}")
    
    # 이 상품의 필드 중에서 새로운 스키마 패턴에 해당하는 것만 찾기
    for field in current_product_fields:
        field_pattern = normalize_field_path(field)
        if field_pattern in new_schema_patterns:
            # 실제 값 추출
            field_path_parts = field.replace(f"product_{product_id}.", "").split(".")
            field_value = current_product
            try:
                for part in field_path_parts:
                    if '[' in part and ']' in part:  # 배열 인덱스 처리
                        key, idx = part.split('[')
                        idx = int(idx.rstrip(']'))
                        if key:
                            field_value = field_value[key][idx]
                        else:
                            field_value = field_value[idx]
                    else:
                        field_value = field_value[part]
            except (KeyError, IndexError, TypeError):
                field_value = None
            
            all_new_field_changes.append({
                "product_id": product_id,
                "field": field,
                "change_type": "added",
                "old_value": None,
                "new_value": field_value
            })

print(f"\n새 스키마/필드 검사 완료:")
print(f"- 진짜 새로운 필드 변경사항: {len(all_new_field_changes)}개")

if all_new_field_changes:
    sample = all_new_field_changes[0]
    print(f"\n샘플 새 필드:")
    print(f"  - 상품 ID: {sample['product_id']}")
    print(f"  - 새 필드: {sample['field']}")
    print(f"  - 새 값: {str(sample['new_value'])[:100]}...")

# 변경사항 결과 정리 (다음 단계에서 사용할 변수들)
changes_result = {
    "timestamp": datetime.now().isoformat(),
    "comparison_info": {
        "previous_file": previous_blob_name or "없음",
        "current_file": current_blob_name,
        "previous_products_count": len(previous_products_dict),
        "current_products_count": len(current_products_dict)
    },
    "change_summary": {
        "total_new_field_changes": len(all_new_field_changes),
        "new_products_count": len(added_ids),
        "removed_products_count": len(removed_ids)
    },
    "new_field_changes": all_new_field_changes
}

detected_changes = changes_result
has_changes = len(all_new_field_changes) > 0

print(f"\n=== 최종 결과 ===")
print(f"- 변경사항 있음: {has_changes}")
print(f"- 총 새 필드 변경사항: {len(all_new_field_changes)}개")

## 6. 변경사항을 GCS에 저장

In [ ]:
saved_change_files = []

if has_changes:
    if not version_date:
        version_date = datetime.now().strftime("%Y%m%d")
    
    # 1. JSON 형식으로 GCS에 저장
    def save_changes_to_gcs(data: Dict[str, Any], filename: str) -> str:
        """변경사항을 GCS에 저장"""
        try:
            gcs_path = f"field_changes/{dt}/{filename}"
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(
                json.dumps(data, ensure_ascii=False, indent=2),
                content_type='application/json'
            )
            print(f"GCS 저장 완료: gs://{gcs_bucket_name}/{gcs_path}")
            return gcs_path
        except Exception as e:
            print(f"GCS 저장 실패 {filename}: {str(e)}")
            return None
    
    # JSON 파일 저장
    json_filename = f"field_changes_{version_date}.json"
    json_gcs_path = save_changes_to_gcs(detected_changes, json_filename)
    if json_gcs_path:
        saved_change_files.append(json_gcs_path)
    
    # 2. 텍스트 요약 저장
    def create_text_summary(changes_data: Dict[str, Any]) -> str:
        """변경사항을 텍스트 요약으로 변환"""
        lines = []
        lines.append("=== 새 필드/스키마 변경사항 요약 ===")
        lines.append(f"타임스탬프: {changes_data.get('timestamp', 'N/A')}")
        lines.append("")
        
        summary = changes_data.get('change_summary', {})
        lines.append("■ 변경 요약:")
        lines.append(f"  - 총 새 필드 변경사항: {summary.get('total_new_field_changes', 0)}개")
        lines.append("")
        
        new_field_changes = changes_data.get('new_field_changes', [])
        
        if new_field_changes:
            lines.append("■ 새 필드 상세:")
            
            # 상품별로 그룹핑하여 표시
            by_product = {}
            for change in new_field_changes:
                product_id = change.get('product_id', 'Unknown')
                if product_id not in by_product:
                    by_product[product_id] = []
                by_product[product_id].append(change)
            
            for product_id, product_changes in list(by_product.items())[:10]:  # 최대 10개 상품만 표시
                lines.append(f"  - {product_id} ({len(product_changes)}개 새 필드)")
                for change in product_changes[:5]:  # 각 상품당 최대 5개 필드만 표시
                    field_name = change.get('field', 'Unknown')
                    field_value = str(change.get('new_value', 'N/A'))[:50] + '...' if len(str(change.get('new_value', 'N/A'))) > 50 else str(change.get('new_value', 'N/A'))
                    lines.append(f"    * {field_name}: {field_value}")
                if len(product_changes) > 5:
                    lines.append(f"    ... 외 {len(product_changes) - 5}개 필드")
            
            if len(by_product) > 10:
                lines.append(f"  ... 외 {len(by_product) - 10}개 상품")
        
        return "\n".join(lines)
    
    # 텍스트 요약을 GCS에 저장
    def save_text_to_gcs(text: str, filename: str) -> str:
        """텍스트 파일을 GCS에 저장"""
        try:
            gcs_path = f"field_changes/{dt}/{filename}"
            blob = bucket.blob(gcs_path)
            blob.upload_from_string(text, content_type='text/plain; charset=utf-8')
            print(f"GCS 저장 완료: gs://{gcs_bucket_name}/{gcs_path}")
            return gcs_path
        except Exception as e:
            print(f"GCS 저장 실패 {filename}: {str(e)}")
            return None
    
    summary_text = create_text_summary(detected_changes)
    txt_filename = f"field_changes_summary_{version_date}.txt"
    txt_gcs_path = save_text_to_gcs(summary_text, txt_filename)
    if txt_gcs_path:
        saved_change_files.append(txt_gcs_path)
    
    print(f"\n총 {len(saved_change_files)}개 변경사항 파일 GCS에 저장 완료")
    
else:
    print("\n변경사항이 없어서 파일을 저장하지 않습니다.")

print(f"\n=== 최종 실행 결과 ===")
print(f"- 변경사항 유무: {has_changes}")
print(f"- 총 새 필드 변경사항: {len(all_new_field_changes)}개")
print(f"- GCS 저장된 파일 수: {len(saved_change_files)}개")
if saved_change_files:
    print(f"- 저장 경로:")
    for file_path in saved_change_files:
        print(f"  * gs://{gcs_bucket_name}/{file_path}")